# Neural Networks and Deep Learning (ECS7026P) - Assignment

**CIFAR-10 Image Classification with Custom Attention-Weighted Convolutional Blocks**

This notebook implements the basic architecture described in the assignment specification:
- K intermediate blocks, each with L parallel convolutional layers weighted by a learned attention vector
- An output block with global average pooling and fully connected layer(s)
- Training and evaluation on CIFAR-10

## 1. Imports and Setup

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
import torchvision
import torchvision.transforms as transforms
from torch.utils.data import DataLoader
import matplotlib.pyplot as plt
import numpy as np
import time

# Device configuration
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Using device: {device}')

## 2. DataLoaders

Load CIFAR-10 with basic transforms. For the initial basic architecture we use minimal
augmentation (just tensor conversion and normalisation). Data augmentation can be added
later in the optimisation phase.

In [ ]:
# CIFAR-10 channel-wise mean and std (precomputed standard values)
CIFAR10_MEAN = (0.4914, 0.4822, 0.4465)
CIFAR10_STD = (0.2470, 0.2435, 0.2616)

# Training transforms: basic normalisation only for now
transform_train = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize(CIFAR10_MEAN, CIFAR10_STD),
])

# Test transforms: same normalisation, no augmentation
transform_test = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize(CIFAR10_MEAN, CIFAR10_STD),
])

# Download and load datasets
train_dataset = torchvision.datasets.CIFAR10(
    root='./data', train=True, download=True, transform=transform_train
)
test_dataset = torchvision.datasets.CIFAR10(
    root='./data', train=False, download=True, transform=transform_test
)

# Create DataLoaders
BATCH_SIZE = 128

train_loader = DataLoader(
    train_dataset, batch_size=BATCH_SIZE, shuffle=True, num_workers=2, pin_memory=True
)
test_loader = DataLoader(
    test_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=2, pin_memory=True
)

print(f'Training samples: {len(train_dataset)}')
print(f'Testing samples:  {len(test_dataset)}')
print(f'Batch size:       {BATCH_SIZE}')
print(f'Training batches: {len(train_loader)}')
print(f'Testing batches:  {len(test_loader)}')

## 3. Model Architecture

### 3.1 Intermediate Block

Each intermediate block contains L parallel convolutional layers. The input image x is:
1. Global-average-pooled across spatial dimensions to produce a c-dimensional vector **m**
2. **m** is passed through a fully connected layer to produce an L-dimensional vector **a**
3. **a** is passed through softmax to produce normalised attention weights
4. x is passed independently through each of the L conv layers
5. The outputs are combined as a weighted sum: x' = a1*C1(x) + a2*C2(x) + ... + aL*CL(x)

In [ ]:
class IntermediateBlock(nn.Module):
    """Attention-weighted parallel convolutional block.
    
    Args:
        in_channels:  Number of input channels.
        out_channels: Number of output channels (same for all L conv layers).
        num_convs:    Number of parallel convolutional layers (L).
        kernel_sizes: List of kernel sizes for each conv layer. If a single int
                      is provided, all conv layers use the same kernel size.
    """
    def __init__(self, in_channels, out_channels, num_convs=3, kernel_sizes=3):
        super().__init__()
        
        self.num_convs = num_convs
        
        # Handle kernel_sizes: allow a single int or a list
        if isinstance(kernel_sizes, int):
            kernel_sizes = [kernel_sizes] * num_convs
        assert len(kernel_sizes) == num_convs, (
            f'Expected {num_convs} kernel sizes, got {len(kernel_sizes)}'
        )
        
        # L independent convolutional layers, each producing (batch, out_channels, H, W)
        # Padding is set to kernel_size // 2 to preserve spatial dimensions
        self.conv_layers = nn.ModuleList([
            nn.Sequential(
                nn.Conv2d(in_channels, out_channels, kernel_size=ks, padding=ks // 2),
                nn.BatchNorm2d(out_channels),
                nn.ReLU(inplace=True),
            )
            for ks in kernel_sizes
        ])
        
        # Fully connected layer to compute attention weights from channel averages
        # Input: c_in dimensional vector (global avg of each channel)
        # Output: L dimensional vector (one weight per conv layer)
        self.attention_fc = nn.Linear(in_channels, num_convs)
    
    def forward(self, x):
        # x shape: (batch, c_in, H, W)
        
        # Step 1: Compute channel-wise global average -> m of shape (batch, c_in)
        m = x.mean(dim=(2, 3))  # average over spatial dimensions H, W
        
        # Step 2: Compute attention weights -> a of shape (batch, L)
        a = self.attention_fc(m)
        a = torch.softmax(a, dim=1)  # normalise so weights sum to 1
        
        # Step 3: Pass x through each conv layer independently
        conv_outputs = [conv(x) for conv in self.conv_layers]  # list of L tensors, each (batch, c_out, H, W)
        
        # Step 4: Stack conv outputs and compute weighted sum
        # Stack: (batch, L, c_out, H, W)
        stacked = torch.stack(conv_outputs, dim=1)
        
        # Reshape a for broadcasting: (batch, L, 1, 1, 1)
        a = a.unsqueeze(2).unsqueeze(3).unsqueeze(4)
        
        # Weighted sum over the L dimension: (batch, c_out, H, W)
        x_out = (a * stacked).sum(dim=1)
        
        return x_out

### 3.2 Output Block

Global average pooling followed by fully connected layer(s) to produce 10-class logits.

In [ ]:
class OutputBlock(nn.Module):
    """Output block: global average pooling -> FC layers -> logits.
    
    Args:
        in_channels: Number of channels in the input image (from last intermediate block).
        num_classes: Number of output classes (10 for CIFAR-10).
        hidden_dim:  If provided, adds a hidden FC layer with this many units before
                     the output layer. If None, maps directly from in_channels to num_classes.
    """
    def __init__(self, in_channels, num_classes=10, hidden_dim=None):
        super().__init__()
        
        if hidden_dim is not None:
            self.fc = nn.Sequential(
                nn.Linear(in_channels, hidden_dim),
                nn.ReLU(inplace=True),
                nn.Linear(hidden_dim, num_classes),
            )
        else:
            self.fc = nn.Linear(in_channels, num_classes)
    
    def forward(self, x):
        # x shape: (batch, c, H, W)
        # Global average pooling over spatial dimensions
        m = x.mean(dim=(2, 3))  # (batch, c)
        o = self.fc(m)           # (batch, num_classes)
        return o

### 3.3 Full Network

Compose K intermediate blocks followed by the output block. MaxPool2d layers are inserted
between blocks to progressively reduce spatial dimensions (32 -> 16 -> 8).

In [ ]:
class CIFAR10Net(nn.Module):
    """Full network: sequence of IntermediateBlocks -> OutputBlock.
    
    Args:
        block_configs: List of dicts, each specifying the config for one IntermediateBlock.
                       Keys: 'in_channels', 'out_channels', 'num_convs', 'kernel_sizes'
        num_classes:   Number of output classes.
        output_hidden: Hidden dimension for the output block (None for direct mapping).
    """
    def __init__(self, block_configs, num_classes=10, output_hidden=None):
        super().__init__()
        
        layers = []
        for i, cfg in enumerate(block_configs):
            layers.append(IntermediateBlock(
                in_channels=cfg['in_channels'],
                out_channels=cfg['out_channels'],
                num_convs=cfg.get('num_convs', 3),
                kernel_sizes=cfg.get('kernel_sizes', 3),
            ))
            # Add max pooling after each block except the last to downsample spatially
            if i < len(block_configs) - 1:
                layers.append(nn.MaxPool2d(kernel_size=2, stride=2))
        
        self.features = nn.Sequential(*layers)
        
        # Output block takes the channel count from the last intermediate block
        last_channels = block_configs[-1]['out_channels']
        self.output_block = OutputBlock(last_channels, num_classes, hidden_dim=output_hidden)
    
    def forward(self, x):
        x = self.features(x)
        o = self.output_block(x)
        return o

### 3.4 Instantiate the Model

Basic configuration: 3 blocks with increasing channel widths (32 -> 64 -> 128),
each with 3 parallel conv layers using 3x3 kernels.

Spatial dimensions after each block:
- Input: 3 x 32 x 32
- Block 1 output: 32 x 32 x 32 -> MaxPool -> 32 x 16 x 16
- Block 2 output: 64 x 16 x 16 -> MaxPool -> 64 x 8 x 8
- Block 3 output: 128 x 8 x 8
- Output block: GAP -> 128 -> FC -> 10

In [ ]:
block_configs = [
    {'in_channels': 3,  'out_channels': 32,  'num_convs': 3, 'kernel_sizes': [3, 3, 3]},
    {'in_channels': 32, 'out_channels': 64,  'num_convs': 3, 'kernel_sizes': [3, 3, 3]},
    {'in_channels': 64, 'out_channels': 128, 'num_convs': 3, 'kernel_sizes': [3, 3, 3]},
]

model = CIFAR10Net(block_configs, num_classes=10, output_hidden=None)
model = model.to(device)

# Verify with a dummy forward pass
dummy_input = torch.randn(2, 3, 32, 32).to(device)
dummy_output = model(dummy_input)
print(f'Input shape:  {dummy_input.shape}')
print(f'Output shape: {dummy_output.shape}')  # should be (2, 10)

# Print model parameter count
total_params = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f'Total parameters:     {total_params:,}')
print(f'Trainable parameters: {trainable_params:,}')

## 4. Training and Evaluation

In [ ]:
def evaluate(model, data_loader, device):
    """Compute accuracy on a dataset."""
    model.eval()
    correct = 0
    total = 0
    with torch.no_grad():
        for images, labels in data_loader:
            images, labels = images.to(device), labels.to(device)
            outputs = model(images)
            _, predicted = outputs.max(dim=1)
            total += labels.size(0)
            correct += predicted.eq(labels).sum().item()
    return 100.0 * correct / total

In [ ]:
# Hyperparameters
NUM_EPOCHS = 30
LEARNING_RATE = 0.01
MOMENTUM = 0.9
WEIGHT_DECAY = 1e-4

# Loss function and optimiser
criterion = nn.CrossEntropyLoss()
optimizer = optim.SGD(
    model.parameters(),
    lr=LEARNING_RATE,
    momentum=MOMENTUM,
    weight_decay=WEIGHT_DECAY,
)

# Learning rate scheduler: reduce LR when progress stalls
scheduler = optim.lr_scheduler.StepLR(optimizer, step_size=10, gamma=0.5)

print(f'Epochs:        {NUM_EPOCHS}')
print(f'Learning rate: {LEARNING_RATE}')
print(f'Optimiser:     SGD (momentum={MOMENTUM}, weight_decay={WEIGHT_DECAY})')

In [ ]:
# Storage for training statistics
batch_losses = []         # loss for each training batch
train_accuracies = []     # training accuracy after each epoch
test_accuracies = []      # testing accuracy after each epoch

# Training loop
for epoch in range(NUM_EPOCHS):
    model.train()
    epoch_start = time.time()
    running_loss = 0.0
    
    for batch_idx, (images, labels) in enumerate(train_loader):
        images, labels = images.to(device), labels.to(device)
        
        # Forward pass
        outputs = model(images)
        loss = criterion(outputs, labels)
        
        # Backward pass and optimise
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        
        # Record batch loss
        batch_losses.append(loss.item())
        running_loss += loss.item()
    
    # Step the learning rate scheduler
    scheduler.step()
    
    # Compute accuracies at the end of each epoch
    train_acc = evaluate(model, train_loader, device)
    test_acc = evaluate(model, test_loader, device)
    train_accuracies.append(train_acc)
    test_accuracies.append(test_acc)
    
    epoch_time = time.time() - epoch_start
    avg_loss = running_loss / len(train_loader)
    current_lr = scheduler.get_last_lr()[0]
    
    print(
        f'Epoch [{epoch+1:2d}/{NUM_EPOCHS}]  '
        f'Loss: {avg_loss:.4f}  '
        f'Train Acc: {train_acc:.2f}%  '
        f'Test Acc: {test_acc:.2f}%  '
        f'LR: {current_lr:.5f}  '
        f'Time: {epoch_time:.1f}s'
    )

print(f'\nBest test accuracy: {max(test_accuracies):.2f}% (epoch {np.argmax(test_accuracies)+1})')

## 5. Plots

The assignment requires:
1. Plot of the loss for each training batch
2. Plot of training accuracy and testing accuracy for each training epoch

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Plot 1: Loss per training batch
axes[0].plot(batch_losses, linewidth=0.5, alpha=0.7)
axes[0].set_xlabel('Batch')
axes[0].set_ylabel('Cross-Entropy Loss')
axes[0].set_title('Training Loss per Batch')
axes[0].grid(True, alpha=0.3)

# Plot 2: Train and test accuracy per epoch
epochs_range = range(1, NUM_EPOCHS + 1)
axes[1].plot(epochs_range, train_accuracies, label='Train Accuracy', marker='o', markersize=4)
axes[1].plot(epochs_range, test_accuracies, label='Test Accuracy', marker='s', markersize=4)
axes[1].set_xlabel('Epoch')
axes[1].set_ylabel('Accuracy (%)')
axes[1].set_title('Training and Testing Accuracy per Epoch')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print(f'Final training accuracy: {train_accuracies[-1]:.2f}%')
print(f'Final testing accuracy:  {test_accuracies[-1]:.2f}%')
print(f'Best testing accuracy:   {max(test_accuracies):.2f}% (epoch {np.argmax(test_accuracies)+1})')

## 6. Summary

### Architecture
- **3 Intermediate Blocks** (K=3), each with 3 parallel conv layers (L=3)
- Channel progression: 3 -> 32 -> 64 -> 128
- All conv layers use 3x3 kernels with padding=1 (spatial-dimension-preserving)
- MaxPool2d (2x2) between blocks for spatial downsampling
- BatchNorm + ReLU after each conv layer
- Attention weights computed via: GAP -> Linear -> Softmax
- **Output Block**: GAP -> Linear(128, 10)

### Training Configuration
- Optimiser: SGD (lr=0.01, momentum=0.9, weight_decay=1e-4)
- Scheduler: StepLR (step_size=10, gamma=0.5)
- Loss: CrossEntropyLoss
- Batch size: 128
- Epochs: 30